In [14]:
#import libraries
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix


# Define the path to the dataset directory
dataset_path = 'rice leaf diseases'

# Define the list of disease types
soil_types = ['Bacterial leaf blight','Brown spot','Leaf smut']

# Define the image size for resizing
img_size = (256, 256)

# Define an empty list to store the image data and target labels
data = []
labels = []

# Loop through each soil type and load the images
for soil_type in soil_types:
    path = os.path.join(dataset_path, soil_type)
    class_num = soil_types.index(soil_type)
    for img in os.listdir(path):
        img_arr = cv2.imread(os.path.join(path, img))
        img_arr = cv2.resize(img_arr, img_size)
        data.append(img_arr)
        labels.append(class_num)

# Convert the data and labels to numpy arrays
data = np.array(data)
labels = np.array(labels)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(data, labels, test_size=0.2)

# Print the shapes of the data and labels
print("X_train shape: ", X_train.shape)
print("y_train shape: ", y_train.shape)
print("X_test shape: ", X_test.shape)
print("y_test shape: ", y_test.shape)


X_train shape:  (288, 256, 256, 3)
y_train shape:  (288,)
X_test shape:  (72, 256, 256, 3)
y_test shape:  (72,)


In [15]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Define the CNN model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(img_size[0], img_size[1], 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(len(soil_types), activation='softmax')
])

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test))

# Evaluate the model on the test data
test_loss, test_acc = model.evaluate(X_test, y_test)
print('Test accuracy:', test_acc)

Epoch 1/10
9/9 [==============================] - 15s 2s/step - loss: 224.3431 - accuracy: 0.4201 - val_loss: 0.9127 - val_accuracy: 0.6389
Epoch 2/10
9/9 [==============================] - 15s 2s/step - loss: 1.1575 - accuracy: 0.6562 - val_loss: 0.7516 - val_accuracy: 0.6111
Epoch 3/10
9/9 [==============================] - 16s 2s/step - loss: 0.5434 - accuracy: 0.7986 - val_loss: 0.7705 - val_accuracy: 0.7083
Epoch 4/10
9/9 [==============================] - 16s 2s/step - loss: 0.8689 - accuracy: 0.7986 - val_loss: 2.0626 - val_accuracy: 0.6667
Epoch 5/10
9/9 [==============================] - 16s 2s/step - loss: 0.7440 - accuracy: 0.8333 - val_loss: 0.4004 - val_accuracy: 0.8889
Epoch 6/10
9/9 [==============================] - 16s 2s/step - loss: 0.2078 - accuracy: 0.9306 - val_loss: 0.6476 - val_accuracy: 0.8611
Epoch 7/10
9/9 [==============================] - 16s 2s/step - loss: 0.1486 - accuracy: 0.9688 - val_loss: 0.5723 - val_accuracy: 0.9583
Epoch 8/10
9/9 [================

In [9]:
from PIL import Image
# Define the list of soil types
soil_types = ['Bacterial leaf blight','Brown spot','Leaf smut']

# Define the image size for resizing
img_size = (256, 256)

# Load the test image and preprocess it
test_img_path = 'Capture.png'
test_img = cv2.imread(test_img_path)
test_img = cv2.resize(test_img, img_size)
test_img = np.expand_dims(test_img, axis=0)

# Make a prediction on the test image
pred = model.predict(test_img)
pred_class = np.argmax(pred)

# Print the predicted soil type
print("Predicted soil type: ", soil_types[pred_class])

error: OpenCV(4.8.1) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\resize.cpp:4062: error: (-215:Assertion failed) !ssize.empty() in function 'cv::resize'


In [17]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Flatten the images into 1D arrays for use with k-NN
X_train_flattened = X_train.reshape((X_train.shape[0], -1))
X_test_flattened = X_test.reshape((X_test.shape[0], -1))

# Initialize a k-NN classifier with k=5
knn = KNeighborsClassifier(n_neighbors=5)

# Fit the classifier to the training data
knn.fit(X_train_flattened, y_train)

# Make predictions on the testing data
y_pred = knn.predict(X_test_flattened)

# Calculate the accuracy of the classifier
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.2f}')

Accuracy: 0.65


In [18]:
from sklearn import svm
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
from skimage import io


# Reshape the image data to flatten each image into a 1-dimensional feature vector
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

# Perform label encoding
le = LabelEncoder()
labels = le.fit_transform(labels)
# Create an SVM classifier
clf = svm.SVC()

# Train the SVM classifier
clf.fit(X_train_flat, y_train)

# Evaluate the classifier on the testing set
y_pred = clf.predict(X_test_flat)

# Convert the classes to strings
target_names = [str(cls) for cls in le.classes_]

# Generate a classification report
report = classification_report(y_test, y_pred, target_names=target_names)
print(report)


              precision    recall  f1-score   support

           0       0.77      0.85      0.81        20
           1       1.00      0.91      0.95        23
           2       0.83      0.83      0.83        29

    accuracy                           0.86        72
   macro avg       0.87      0.86      0.86        72
weighted avg       0.87      0.86      0.86        72



In [12]:
from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier(n_estimators=100, random_state=42)
classifier.fit(X_train_flat, y_train)
y_pred = classifier.predict(X_test_flat)

# Print classification report and confusion matrix
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00         4
           1       0.78      0.88      0.82         8
           2       0.91      0.83      0.87        12

    accuracy                           0.88        24
   macro avg       0.90      0.90      0.90        24
weighted avg       0.88      0.88      0.88        24

[[ 4  0  0]
 [ 0  7  1]
 [ 0  2 10]]
